In [1]:
import sys
path='/home/tomas/Ulmer-Berechnung/alps2qutipplus-april/alps2qutipplus-main/'

sys.path.insert(1, path) 

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import qutip as qutip
import scipy.linalg as linalg
import time
from multiprocessing import Pool
from itertools import product
from typing import Optional

from alpsqutip import (build_system, list_models_in_alps_xml,
                       list_geometries_in_alps_xml, graph_from_alps_xml,
                       model_from_alps_xml,
                       restricted_maxent_toolkit as me)

from alpsqutip.operators.states.utils import safe_exp_and_normalize ## function used to safely and robustly map K-states to states

from alpsqutip.operators.states.meanfield.projections import (one_body_from_qutip_operator, 
                                                  project_operator_to_m_body, 
                                                             project_qutip_operator_to_m_body,
                                                             project_to_n_body_operator,
                                                              project_product_operator_as_n_body_operator,
                                                             project_qutip_operator_as_n_body_operator)

from alpsqutip.operators import (
    ScalarOperator,
    LocalOperator,
    OneBodyOperator,
    Operator,
    ProductOperator,
    ScalarOperator,
    SumOperator,
    QutipOperator
)

from alpsqutip.scalarprod import gram_matrix, fetch_covar_scalar_product, orthogonalize_basis, orthogonalize_basis_gs
from alpsqutip.operators.states.gibbs import GibbsDensityOperator, GibbsProductDensityOperator

In [3]:
params={}

params['size']=7
params['Jx']=1.; params['Jy'] = .75*params['Jx']; params['Jz']=1.05*params['Jx']

from scipy.optimize import root, fsolve
Ffactor=np.real(max(np.roots(np.poly1d([1, 0, -(params['Jx']*params['Jy']+params['Jx']*params['Jy']+params['Jy']*params['Jz']), 
                           -2*params['Jx']*params['Jy']*params['Jz']]))))
chi_y=fsolve(lambda x,y: x*np.arcsinh(x)-np.sqrt(x**2+1)-y, 1e-1, args=(0))[0]
vLR=4*Ffactor*chi_y

In [4]:

system=build_system(geometry_name= "open chain lattice",model_name="spin", 
                    L=params['size'], J=1)

sites=[s for s in system.sites]
sx_ops=[system.site_operator("Sx", '1[' + str(a) + ']') for a in range(len(system.sites))]
sy_ops=[system.site_operator("Sy", '1[' + str(a) + ']') for a in range(len(system.sites))]
sz_ops=[system.site_operator("Sz", '1[' + str(a) + ']') for a in range(len(system.sites))]

idop = [system.site_operator('identity@1[' + str(a) + ']') for a in range(len(system.sites))]
idop = idop[0]*idop[1]*idop[2]*idop[3]*idop[4]#*idop[5]*idop[6]

Iz = sum(sz_ops)
H_nn = 0  # Proches voisins
H_lr = 0  # Longue portée

from itertools import combinations

for i, j in combinations(range(params['size']), 2):
    r = abs(i - j)
    Jx_ij = params['Jx'] / r**3
    Jy_ij = params['Jy'] / r**3
    Jz_ij = params['Jz'] / r**3

    term = (
        Jx_ij * sx_ops[i] * sx_ops[j]
        + Jy_ij * sy_ops[i] * sy_ops[j]
        + Jz_ij * sz_ops[i] * sz_ops[j]
    )

    if r == 1:
        H_nn += term  # Interaction de voisins immédiats
    else:
        H_lr += term  # Interaction à longue portée
        
H = H_nn + H_lr
H = H.simplify()

loading model spin  over graph open chain lattice


In [5]:
HBB0=[idop, sz_ops[3], H_nn]


phi0 = np.array([.0, -3, .10])

K0 = me.k_state_from_phi_basis(phi0, HBB0)
sigma0 = GibbsDensityOperator(K0)
phi0[0] = np.log(sigma0.tr())
K0 = me.k_state_from_phi_basis(phi0, HBB0).simplify()
sigma0 = GibbsDensityOperator(K0)

timespan=np.linspace(.0, 500.1/vLR,75)
obs_SzA = sum(sz for sz in sz_ops)

[(sigma0 * op).tr() for op in sz_ops] 

[(-8.31320488807203e-06+0j),
 (0.0003148904616531844+0j),
 (-0.011931540630634216+0j),
 (0.45243318770778307+0j),
 (-0.011931540630634174+0j),
 (0.00031489046165318266+0j),
 (-8.31320488794713e-06+0j)]

In [6]:
HBBfinal = [idop, sum(sz_ops)]


phif = np.array([.0, -.1])

Kf = me.k_state_from_phi_basis(phif, HBBfinal)
sigmaf = GibbsProductDensityOperator(Kf)
phi0[0] = np.log(sigma0.tr())
Kf = me.k_state_from_phi_basis(phif, HBBfinal).simplify()
sigmaf = GibbsProductDensityOperator(Kf)

obs_SzA = sum(sz for sz in sz_ops)

[(sigmaf * op).tr() for op in sz_ops] 

[0.02497918747894004,
 0.02497918747894004,
 0.02497918747894004,
 0.02497918747894004,
 0.02497918747894004,
 0.02497918747894004,
 0.02497918747894004]

In [7]:
tgt_obs=obs_SzA
test = tgt_obs + 1j*me.commutator(H, tgt_obs) + 1j*me.commutator(H, 1j*me.commutator(H, tgt_obs))

res1 = project_operator_to_m_body(test, 3, sigmaf)

In [8]:
from itertools import product, combinations_with_replacement
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
from functools import partial

# Fonction globale pour que multiprocessing puisse la sérialiser

def _project_single_term(term_m_sigma):
    term, m_max, sigma_0 = term_m_sigma
    return parallel_project_operator_to_m_body(term, m_max, sigma_0, parallel=True)

def _commutator_term_worker(hi_kj):
    hi, kj = hi_kj
    return (hi * kj - kj * hi).simplify().tidyup(1e-5) 

def _sp_worker(pair, basis, sp):
    i, j = pair
    val = float(np.real(sp(basis[i], basis[j])))
    return (i, j, val)

def hij_worker_explicit(args):
    i, basis_i, b_last, H, sp, sigma_0, m_max = args
    # Compute commutator
    comm = (-1j * H * b_last + 1j * b_last * H).simplify().tidyup(1e-5)
    # Project commutator to m-body operators
    comm_proj = parallel_project_operator_to_m_body(
        full_operator = comm, 
        m_max = m_max,
        sigma_0 = sigma_0
    )
    # Compute inner product (b_i | projected commutator)
    val = sp(basis_i, comm_proj).real
    return i, val

def parallel_project_operator_to_m_body(full_operator, m_max=2, sigma_0=None, parallel=True):
    """
    Project a Operator onto a m_max-body operators sub-algebra
    relative to the local state `sigma_0`.
    If `sigma_0` is not given, maximally mixed states are assumed.
    """
    assert sigma_0 is None or hasattr(sigma_0, "expect"), f"{type(sigma_0)} invalid"
    if m_max == 0:
        if sigma_0:
            return ScalarOperator(sigma_0.expect(full_operator), full_operator.system)
        return ScalarOperator(full_operator.tr(), full_operator.system)

    if isinstance(full_operator, OneBodyOperator) or len(full_operator.acts_over()) <= m_max:
        return full_operator

    full_operator = full_operator.simplify()
    system = full_operator.system

    ### parallelization here
    if isinstance(full_operator, SumOperator):
        if parallel and len(full_operator.terms) > 1:
            with ProcessPoolExecutor() as executor:
                args = [(term, m_max, sigma_0) for term in full_operator.terms]
                terms = tuple(executor.map(_project_single_term, args))
        else:
            terms = tuple(
                project_operator_to_m_body(term, m_max, sigma_0, parallel=False)
                for term in full_operator.terms
            )
        if len(terms) == 0:
            return ScalarOperator(0, system)
        if len(terms) == 1:
            return terms[0]
        if len(full_operator.terms) == len(terms) and all(t1 is t2 for t1, t2 in zip(full_operator.terms, terms)):
            return full_operator
        return SumOperator(terms, system).simplify()

    if isinstance(full_operator, ProductOperator):
        sites_op = full_operator.sites_op
        if len(sites_op) <= m_max:
            return full_operator

        first_site, *rest = tuple(sites_op)
        op_first = sites_op[first_site]
        weight_first = op_first
        sigma_rest = sigma_0
        if sigma_0 is not None:
            sigma_rest = sigma_rest.partial_trace(frozenset(rest))
            sigma_first = sigma_0.partial_trace(frozenset({first_site})).to_qutip()
            weight_first = op_first * sigma_first
        else:
            weight_first = weight_first / op_first.dims[0][0]

        first_av = weight_first.tr()
        delta_op = LocalOperator(first_site, op_first - first_av, system)
        sites_op_rest = {site: op for site, op in sites_op.items() if site != first_site}
        rest_prod_operator = ProductOperator(
            sites_op_rest, prefactor=full_operator.prefactor, system=system
        )

        result = delta_op * project_operator_to_m_body(rest_prod_operator, m_max - 1, sigma_rest, parallel)
        if first_av:
            result = result + first_av * project_operator_to_m_body(rest_prod_operator, m_max, sigma_rest, parallel)
        return result.simplify()

    if isinstance(full_operator, QutipOperator):
        return project_qutip_operator_to_m_body(full_operator, m_max, sigma_0)

    return project_qutip_operator_to_m_body(full_operator.to_qutip_operator(), m_max, sigma_0)

def parallelized_real_time_projection_of_hierarchical_basis(generator, 
                                               seed_op,
                                               sigma_ref,
                                               nmax, 
                                               deep,
                                               num_workers=None,
                                               tidy_thresh=1e-5):

    if seed_op is None or deep == 0:
        return []

    basis = [seed_op]
    
    gen_terms = generator.as_sum_of_products().terms

    for i in range(1, deep):
        # Cache last basis terms only once
        basis_last_terms = basis[-1].as_sum_of_products().terms
        term_pairs = list(product(gen_terms, basis_last_terms))

        # Parallelize acá 
        with ProcessPoolExecutor(max_workers=num_workers) as executor:
            commutator_terms = list(executor.map(_commutator_term_worker, term_pairs, chunksize=128))

        # Assemble everything in the local op
        local_op = -1j * sum(commutator_terms)
        if True: 
            local_op = parallel_project_operator_to_m_body(
                            full_operator=local_op.tidyup(1e-5).as_sum_of_products(),
                            m_max=nmax, 
                            sigma_0=sigma_ref,
                            parallel=True
                        ).simplify().tidyup(tidy_thresh)
            
        basis.append(local_op)

    return basis

def parallel_gram_matrix(basis, sp, num_workers=None, use_threads=False):
    """
    Computes the Gram matrix in parallel using a given scalar product.

    Parameters:
        basis: List of operator basis elements.
        sp: Scalar product function, must be top-level and pickleable.
        num_workers: Number of worker threads or processes?
        use_threads: Use threads instead of processes 

    Returns:
        Symmetric Gram matrix as np.ndarray.
    """
    size = len(basis)
    result = np.zeros((size, size), dtype=float)

    executor_cls = ThreadPoolExecutor if use_threads else ProcessPoolExecutor
    index_pairs = list(combinations_with_replacement(range(size), 2))

    # Pre-bind basis and sp to _sp_worker
    worker = partial(_sp_worker, basis=basis, sp=sp)

    with executor_cls(max_workers=num_workers) as executor:
        for i, j, val in executor.map(worker, index_pairs):
            result[i, j] = val
            if i != j:
                result[j, i] = val

    return result.round(14)

def orthogonalize_basis_gs_parallel(basis, sp: callable, tol=1e-5, num_threads=4):
    """
    Orthogonalizes a given basis using Gram-Schmidt with scalar product `sp`.
    Uses threading to parallelize the inner scalar products where possible.

    Parameters:
        basis: list of operators or matrices.
        sp: scalar product function.
        tol: discard vectors with norm < tol.
        num_threads: number of threads to use in parallel regions.

    Returns:
        orth_basis: orthonormalized basis.
    """
    orth_basis = []

    for op_orig in basis:
        op = op_orig
        
        # Medio que en Gram Schmidt mucho no puedo paralelizar 
        # solo las proyecciones del (op_a, op_b) * op_b
        with ThreadPoolExecutor(max_workers=num_threads) as executor:
            projections = list(executor.map(
                lambda prev: sp(prev, op),
                orth_basis
            ))

        for proj, prev in zip(projections, orth_basis):
            op -= proj * prev

        norm = np.real(sp(op, op)) ** 0.5
        if norm < tol:
            continue
        op /= norm
        orth_basis.append(op)

    # Parallel orthonormality check (optional)
    def check_orthonorm(pair):
        i, j = pair
        val = sp(orth_basis[i], orth_basis[j])
        if i == j:
            assert abs(val - 1.0) < tol, f"Norm not 1 at {i}: {val}"
        else:
            assert abs(val) < tol, f"Not orthogonal: {i}, {j} = {val}"

    with ThreadPoolExecutor(max_workers=num_threads) as executor:
        executor.map(check_orthonorm, [(i, j) for i in range(len(orth_basis)) for j in range(i, len(orth_basis))])

    return orth_basis

def parallel_Hij_tensor(basis, generator, sigma_0, m_max, use_threads=False, max_workers=None):
    """
    Computes Hij = (b_i, [H, b_j]) for all i,j using the known basis structure:
      - For j < ell-1, [H, b_j] = approx b_{j+1}
      - For j = ell-1, compute and project [H, b_j] explicitly
    """
    ell = len(basis)
    Hij = np.zeros((ell, ell), dtype=np.float64)
    sp_local = fetch_covar_scalar_product(sigma_0)
    
    # Case 1: j < ell - 1 → use (b_i, b_{j+1})
    for i in range(ell):
        for j in range(ell - 1):  # skip last column
            Hij[i, j] = sp_local(basis[i], basis[j + 1]).real

    # Case 2: j = ell - 1 → compute [H, b_{ell-1}] explicitly
    executor_cls = ThreadPoolExecutor if use_threads else ProcessPoolExecutor
    tasks = [(i, basis[i], basis[-1], generator, sp_local, sigma_0, m_max) for i in range(ell)]

    with executor_cls(max_workers=max_workers) as executor:
        for i, val in executor.map(hij_worker_explicit, tasks):
            Hij[i, ell - 1] = val

    return Hij


In [11]:
start = time.time()
sp = fetch_covar_scalar_product(sigmaf)
HBB_ell_act = parallelized_real_time_projection_of_hierarchical_basis(generator = H,
                                                                 seed_op = tgt_obs,
                                                                 sigma_ref = sigmaf,
                                                                 nmax = 3,
                                                                 deep = 5)

print(time.time() - start); start = time.time()

8.300391674041748


In [20]:
start = time.time()

Hij_tensor_act = parallel_Hij_tensor(HBB_ell_act, H, sigmaf, m_max = 2, max_workers=8)

print(time.time()-start)

80.13691234588623


In [21]:
Hij_tensor_act ### me comí un menos que era un más?

array([[ 0.        , -0.04823996,  0.        ,  0.04359757,  0.        ],
       [ 0.04765375,  0.        , -0.04293513,  0.        ,  0.10294693],
       [ 0.        ,  0.04295213,  0.        , -0.10295778,  0.        ],
       [-0.04293513,  0.        ,  0.10294355,  0.        , -0.30740416],
       [ 0.        , -0.10295778,  0.        ,  0.30947434,  0.        ]])